# scVI best - cellranger_versions

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import inspect
import json
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/scvi_py310/lib/R"
)
os.environ["R_LIBS_SITE"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R/library"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib")

OPTIMIZER_PATH = Path(
    PROJECT_ROOT,
    "scripts",
    "optuna_multi_metrics",
    "optuna_scvi",
    "optimize_scvi_multi_metrics.py",
)
sys.path.insert(0, str(OPTIMIZER_PATH.parent))

import anndata as ad
import numpy as np
import pandas as pd
import torch
import optimize_scvi_multi_metrics as scvi_workflow

assert Path(scvi_workflow.__file__).resolve() == OPTIMIZER_PATH


In [2]:
DATASET_NAME = 'cellranger_versions'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEYS = ('batch',)
BATCH_KEY = BATCH_KEYS[0]
LABEL_KEY = 'cell_type'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
USE_HVG = True
N_TOP_HVG = 2000
MAX_EPOCHS = 400

EMBED_KEY = "X_scvi"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scvi_metric_batch"
METRIC_LABEL_KEY = "scvi_metric_label"

assert DATASET_NAME == 'cellranger_versions'
assert BATCH_KEYS == ('batch',)
assert LABEL_KEY == 'cell_type'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_keys": list(BATCH_KEYS),
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
        "max_epochs": MAX_EPOCHS,
    },
    name="configuration",
)


dataset                                     cellranger_versions
data_path     /ubc/cs/research/beaver/projects/Single-cell-i...
batch_keys                                              [batch]
label_key                                             cell_type
metrics                [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
use_hvg                                                    True
n_top_hvg                                                  2000
max_epochs                                                  400
Name: configuration, dtype: object

In [3]:
scvi_workflow.DATA_PATH = DATA_PATH
scvi_workflow.DATASET_NAME = DATASET_NAME
scvi_workflow.BATCH_KEYS = list(BATCH_KEYS)
scvi_workflow.BATCH_KEY = BATCH_KEY
scvi_workflow.LABEL_KEY = LABEL_KEY
scvi_workflow.USE_HVG = USE_HVG
scvi_workflow.N_TOP_HVG = N_TOP_HVG
scvi_workflow.MAX_EPOCHS = MAX_EPOCHS
scvi_workflow.N_NEIGHBORS = N_NEIGHBORS
scvi_workflow.EMBED_KEY = EMBED_KEY
scvi_workflow.CLUSTER_KEY = CLUSTER_KEY
scvi_workflow.METRIC_BATCH_KEY = METRIC_BATCH_KEY
scvi_workflow.METRIC_LABEL_KEY = METRIC_LABEL_KEY

scvi_workflow.USE_BAMI = "BAMI" in SELECTED_METRICS
scvi_workflow.USE_CASW = "CASW" in SELECTED_METRICS
scvi_workflow.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
scvi_workflow.USE_TRAJ = "TRAJ" in SELECTED_METRICS
scvi_workflow.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
scvi_workflow.USE_BRAS = "BRAS" in SELECTED_METRICS
scvi_workflow.USE_KBET = "KBET" in SELECTED_METRICS
scvi_workflow.USE_CPCR = "CPCR" in SELECTED_METRICS

scvi_workflow.validate_configuration(SELECTED_METRICS)
scvi_workflow.validate_dataset_contents()

pd.Series(
    {
        "optimizer": str(OPTIMIZER_PATH),
        "scvi_version": scvi_workflow.scvi.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    name="preflight",
)


Dataset preflight: n_obs=19202 n_batches=2 n_labels=15 counts_layer=True
CPCR preflight: eligible_labels=12


optimizer         /ubc/cs/research/beaver/projects/Single-cell-i...
scvi_version                                                  1.3.3
cuda_available                                                 True
Name: preflight, dtype: object

In [4]:
OPTUNA_BEST_SOURCE = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/optuna_multi_metrics/scvi/cellranger_versions/expanded_metrics_v1/scvi_cellranger_versions_joint_expanded_metrics_v1_best_params.json'
OPTUNA_TRIALS_SOURCE = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/optuna_multi_metrics/scvi/cellranger_versions/expanded_metrics_v1/scvi_cellranger_versions_joint_expanded_metrics_v1_trials.csv'
OPTUNA_STUDY_NAME = 'scvi_cellranger_versions_joint_expanded_metrics_v1'
OPTUNA_BEST_TRIAL = 193
OPTUNA_BEST_VALUE = 0.8135800655625763
OPTUNA_BEST_PARAMS = {'n_latent': 61, 'n_hidden': 128, 'n_layers': 3, 'dropout_rate': 0.020039543742391263, 'lr': 0.0012313512408809633}

with open(OPTUNA_BEST_SOURCE, "r", encoding="utf-8") as source_file:
    current_source = json.load(source_file)
assert current_source["study_name"] == OPTUNA_STUDY_NAME
assert current_source["dataset_name"] == DATASET_NAME
assert current_source["best_value"] == OPTUNA_BEST_VALUE
assert current_source["best_params"] == OPTUNA_BEST_PARAMS
assert tuple(OPTUNA_BEST_PARAMS) == ('n_latent', 'n_hidden', 'n_layers', 'dropout_rate', 'lr')

SCVI_PARAMS = dict(OPTUNA_BEST_PARAMS)

pd.Series(
    {
        "source": OPTUNA_BEST_SOURCE,
        "trials_source": OPTUNA_TRIALS_SOURCE,
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **SCVI_PARAMS,
        "gene_likelihood": "zinb",
        "dispersion": "gene",
        "latent_distribution": "normal",
        "early_stopping": True,
        "max_epochs": MAX_EPOCHS,
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
    },
    name="best_setting",
)

def run_scvi_setting(adata, seed):
    return scvi_workflow.run_scvi(
        adata,
        n_latent=SCVI_PARAMS["n_latent"],
        n_hidden=SCVI_PARAMS["n_hidden"],
        n_layers=SCVI_PARAMS["n_layers"],
        dropout_rate=SCVI_PARAMS["dropout_rate"],
        lr=SCVI_PARAMS["lr"],
        seed=int(seed),
    )


In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    result = scvi_workflow.compute_joint_metrics(adata_raw, adata_int)
    if tuple(result) != (
        *SELECTED_METRICS,
        "bio_score",
        "batch_score",
        "joint_score",
    ):
        raise ValueError(f"Unexpected metric result keys: {tuple(result)}")
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scvi_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


[rank: 0] Seed set to 0


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/scvi_py310/lib/python3.10/site-packages/scvi/dataloaders/_data_splitting.py:258: UserWarning: 2 cells moved from training set to validation set. if you want to avoid it please use train_size parameter during train.
  self.n_train, self.n_val = validate_data_split(
GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.783466057456778


optimised clustering against scvi_metric_label
optimal cluster resolution: 0.8
optimal score: 0.8058834004057893
BRAS...


seed           9.000000
BAMI           0.805883
CASW           0.543163
SCGRAPHM       0.788007
BRAS           0.942207
KBET           0.791643
CPCR           0.986217
bio_score      0.712351
batch_score    0.906689
joint_score    0.790086
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.830406,0.551978,0.835714,0.950808,0.836059,0.987837,0.739366,0.924901,0.813580
1,1,0.829822,0.551825,0.767548,0.932395,0.784724,0.980823,0.716398,0.899314,0.789565
2,2,0.825771,0.550729,0.795265,0.946457,0.828551,0.986224,0.723922,0.920411,0.802517
3,3,0.816999,0.551280,0.799170,0.935305,0.777205,0.977900,0.722483,0.896804,0.792211
4,4,0.837718,0.555680,0.821714,0.933413,0.784106,0.980889,0.738370,0.899469,0.802810
5,5,0.814734,0.551412,0.783174,0.944274,0.820948,0.987316,0.716440,0.917512,0.796869
6,6,0.809238,0.548567,0.788280,0.948594,0.842103,0.988550,0.715362,0.926416,0.799783
7,7,0.809119,0.542004,0.796810,0.947225,0.806770,0.991855,0.715977,0.915283,0.795700
8,8,0.821566,0.550009,0.778036,0.943546,0.850620,0.990038,0.716537,0.928068,0.801149
9,9,0.805883,0.543163,0.788007,0.942207,0.791643,0.986217,0.712351,0.906689,0.790086


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.820126,0.010122,0.805883,0.837718
CASW,0.549665,0.003941,0.542004,0.555680
SCGRAPHM,0.795372,0.019147,0.767548,0.835714
BRAS,0.942422,0.006204,0.932395,0.950808
KBET,0.812273,0.025503,0.777205,0.850620
CPCR,0.985765,0.004246,0.977900,0.991855
bio_score,0.721721,0.009156,0.712351,0.739366
batch_score,0.913487,0.011399,0.896804,0.928068
joint_score,0.798427,0.006866,0.789565,0.813580
